# Redis Vector DB: Features and Usage in Python

This notebook demonstrates all major features of Redis Vector Database and their usage in Python, including:
- Database and collection creation
- Adding data with embeddings
- Metadata handling
- Embedding functions (Sentence Transformers, OpenAI)
- Advanced features: hybrid search, reranking, multimodal data



## Install and Import Required Libraries

- Redis Vector DB, embeddings, and advanced features.

In [ ]:
#!pip install redis

In [ ]:
#!pip install redis-py-cluster

  Using cached redis-3.5.3-py2.py3-none-any.whl.metadata (36 kB)
Using cached redis-3.5.3-py2.py3-none-any.whl (72 kB)
  Attempting uninstall: redis
    Found existing installation: redis 4.5.5
    Uninstalling redis-4.5.5:
      Successfully uninstalled redis-4.5.5


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langflow 1.6.9 requires redis<6.0.0,>=5.2.1, but you have redis 3.5.3 which is incompatible.
redisvl 0.12.1 requires redis<7.0,>=5.0, but you have redis 3.5.3 which is incompatible.


In [ ]:
#!pip install redis redis-py-cluster sentence-transformers openai numpy pillow

In [14]:
# Install required packages (uncomment if running for the first time)
# !pip install redis redis-py-cluster sentence-transformers openai numpy pillow

import redis
import numpy as np
from sentence_transformers import SentenceTransformer
import openai
from PIL import Image
import io
import os

# For advanced features
import random
import json

### Redis Internal Features & Architecture
- **In-memory store:** All data is kept in RAM for ultra-fast access; persistence is optional (RDB/AOF).
- **Key-value model:** Data is stored as key-value pairs; values can be strings, hashes, lists, sets, sorted sets, streams, or modules (like vectors).
- **No schema:** Flexible, schema-less; you define your own structure via key naming conventions.
- **Modules:** Extensible via modules (e.g., Redisearch for vector search, JSON, etc.).
- **Indexes:** Redisearch provides secondary indexes and vector search capabilities.
- **Single-threaded core:** Handles requests sequentially (but modules and I/O can be multi-threaded).

### Postgres Internal Features & Architecture
- **Disk-based RDBMS:** Data is stored on disk, with caching for performance.
- **Relational model:** Data is organized in tables with a fixed schema (columns, types).
- **ACID compliance:** Strong transactional guarantees.
- **Extensions:** Can be extended (e.g., pgvector for vector data, PostGIS for geospatial).
- **Indexes:** Supports B-tree, GIN, GiST, and HNSW (via pgvector) for fast search.
- **Multi-process:** Uses multiple processes for concurrency and parallelism.

---

### Handling Vector Data

| Feature                | Redis (with Redisearch)         | Postgres (with pgvector)         |
|------------------------|---------------------------------|----------------------------------|
| Data Model             | Key-value, schema-less          | Tables, schema-based             |
| Vector Storage         | As binary blobs in hashes/JSON  | As vector column in a table      |
| Indexing               | HNSW, FLAT (via Redisearch)     | HNSW, IVFFlat (via pgvector)     |
| Query                  | KNN, hybrid (vector+metadata)   | KNN, hybrid (vector+SQL filter)  |
| Performance            | Very fast (in-memory)           | Fast, but disk-based             |
| Scalability            | Easy to scale horizontally      | Scales vertically, some horiz.   |
| ACID Transactions      | Limited                         | Full ACID                        |
| Use Case Fit           | Real-time, low-latency search   | Analytical, transactional, hybrid|

---

### Similarities
- Both can store and search vector data with KNN and hybrid queries.
- Both support extensions/modules for vector search.

### Dissimilarities
- Redis is in-memory and schema-less; Postgres is disk-based and schema-driven.
- Redis is optimized for speed and real-time workloads; Postgres is optimized for reliability and complex queries.
- Redis uses key prefixes and modules; Postgres uses tables and extensions.

## Connect to Redis Vector Database

Establish a connection to your local Redis instance. Make sure your Redis server (with Redisearch) is running.

In [15]:
# Connect to Redis
redis_host = 'localhost'
redis_port = 6379
redis_password = None  # Set if you have a password

try:
    r = redis.Redis(host=redis_host, port=redis_port, password=redis_password, decode_responses=True)
    r.ping()
    print('Connected to Redis!')
    # Print additional DB info
    print("------------------------------")
    print('Redis server info:')
    info = r.info()
    print(f"Redis version: {info.get('redis_version')}")
    print(f"Connected clients: {info.get('connected_clients')}")
    print(f"Used memory: {info.get('used_memory_human')}")
    print(f"Total keys: {r.dbsize()}")
    print(f"Uptime (days): {info.get('uptime_in_days')}")
    # Print number of databases
    db_count = 0
    for key in info:
        if key.startswith('db') and key[2:].isdigit():
            db_count += 1
    print(f"Number of databases: {db_count}")
    print("------------------------------")
except Exception as e:
    print(f'Error connecting to Redis: {e}')

Connected to Redis!
------------------------------
Redis server info:
Redis version: 7.4.7
Connected clients: 2
Used memory: 930.32M
Total keys: 25000
Uptime (days): 0
Number of databases: 1
------------------------------


## Create a Redis Database (Namespace)

Redis does not have traditional databases or namespaces, but you can use key prefixes to simulate namespaces for vector search.

### Why does RedisInsight (GUI) have an "Add a database" button?

- In Redis, the term "database" is different from relational databases. Redis supports multiple logical databases (numbered 0, 1, 2, ...), which are just isolated key spaces within the same server instance—not true databases with separate users, schemas, or permissions.
- By default, most clients connect to database 0.
- The "Add a database" button in GUI tools (like RedisInsight) is for convenience. It lets you:
  - Connect to different Redis servers/instances, or
  - Switch to a different logical database number within the same server.
- These are not like SQL databases or MongoDB collections—they’re just separate key spaces for organizing data.

**Summary:**
- Redis "databases" are logical partitions, not full-featured databases.
- The GUI button helps you manage connections and logical DBs, but the underlying architecture is still key-value and schema-less.

In [16]:
# Define a namespace (key prefix) for your vector data
NAMESPACE = 'vectordb:'

# Example usage: all keys will be prefixed with 'vectordb:'

## Install redis vector library (pip install -U redisvl)

RedisVL (Redis Vector Library) is a Python library that provides a high-level interface for working with vector data in Redis, especially for AI and machine learning use cases. It simplifies storing, indexing, and searching vector embeddings in Redis, and is often used with frameworks like LangChain.

Key points:

- Makes it easier to use Redis as a vector database from Python.
- Handles vector storage, indexing, and similarity search.
- Integrates with popular embedding models and frameworks.

In [17]:
!pip install -U redisvl

### Schema in RedisVL

The schema in RedisVL defines the structure of your vector index in Redis. It specifies what fields (such as vectors and metadata) are present, their types, and how they are indexed and searched.

**Key Points:**
- The schema is created using the `IndexSchema` class.
- You define fields such as text, tag, numeric, and vector fields.
- Vector fields specify the dimension, data type (e.g., FLOAT32), and the algorithm (e.g., HNSW, FLAT) for similarity search.
- You can also configure stopwords and advanced vector index settings.

**Example:**
```python
from redisvl.schema import IndexSchema, TextField, VectorField

schema = IndexSchema(
    fields=[
        TextField("title"),
        VectorField("embedding", dims=1536, algorithm="HNSW", datatype="FLOAT32")
    ]
)
```
This schema defines an index with a text field called `title` and a vector field called `embedding` (1536 dimensions, HNSW algorithm).

The schema is essential for telling RedisVL how to store, index, and search your data efficiently.

#### Schema Components in RedisVL

| Component | Description |
|-----------|-------------|
| **version** | The version of the schema specification. The current supported version is `0.1.0`. |
| **index** | Index-specific settings such as name, key prefix, key separator, and storage type. |
| **fields** | The subset of fields within your data to include in the index, along with any custom settings for each field. |

These components allow you to precisely define how your data is structured, indexed, and queried in RedisVL.

#### IndexSchema Example in RedisVL

The `IndexSchema` class in RedisVL defines the structure and configuration for a search index in Redis, organizing both vector and metadata fields. 

You can create an `IndexSchema` from a Python dictionary or a YAML file, making it flexible for different workflows.

**Example YAML schema:**
```yaml
version: '0.1.0'

index:
  name: user-index
  prefix: user
  key_separator: ":"
  storage_type: json

fields:
  - name: user
    type: tag
  - name: credit_score
    type: tag
  - name: embedding
    type: vector
    attrs:
      algorithm: flat
      dims: 3
      distance_metric: cosine
      datatype: float32
```

This schema defines:
- An index named `user-index` with keys prefixed by `user:`
- Storage type as JSON
- Two tag fields (`user`, `credit_score`)
- One vector field (`embedding`) with 3 dimensions, using the FLAT algorithm, cosine distance, and float32 data type

You can load this schema in Python and use it to create or manage your Redis vector index.

In [21]:
!pip install redis==4.5.5

  Attempting uninstall: redis
    Found existing installation: redis 6.4.0
    Uninstalling redis-6.4.0:
      Successfully uninstalled redis-6.4.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langflow 1.6.9 requires redis<6.0.0,>=5.2.1, but you have redis 4.5.5 which is incompatible.
redisvl 0.12.1 requires redis<7.0,>=5.0, but you have redis 4.5.5 which is incompatible.
redis-py-cluster 2.1.3 requires redis<4.0.0,>=3.0.0, but you have redis 4.5.5 which is incompatible.


In [2]:
from redisvl.schema import IndexSchema

schema = IndexSchema.from_yaml("schema.yaml")

... Loading the schema for RedisVL from dict

In [6]:
schema = IndexSchema.from_dict({
    "index": {
        "name":             "user-index",
        "prefix":           "user",
        "key_separator":    ":",
        "storage_type":     "json",
    },
    "fields": [
        {"name": "user", "type": "tag"},
        {"name": "credit_score", "type": "tag"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "algorithm": "flat",
                "dims": 3,
                "distance_metric": "cosine",
                "datatype": "float32"
            }
        }
    ]
})

> Note

The fields attribute in the schema must contain unique field names to ensure correct and unambiguous field references.

#### Add field to IndexSchema

Extends the schema with additional fields.

This method allows dynamically adding new fields to the index schema. It processes a list of field definitions.

In [7]:
# Add a tag field

try:
    schema.add_field({"name": "user1", "type": "tag"})
    print("Tag field 'user1' added to schema.")
except Exception as e:
    print(f"Error : {e}")

Tag field 'user1' added to schema.


In [4]:
print(schema)

index=IndexInfo(name='user-index', prefix='user', key_separator=':', storage_type=<StorageType.JSON: 'json'>, stopwords=None) fields={'user': TagField(name='user', type=<FieldTypes.TAG: 'tag'>, path='$.user', attrs=TagFieldAttributes(sortable=False, index_missing=False, no_index=False, separator=',', case_sensitive=False, withsuffixtrie=False, index_empty=False)), 'credit_score': TagField(name='credit_score', type=<FieldTypes.TAG: 'tag'>, path='$.credit_score', attrs=TagFieldAttributes(sortable=False, index_missing=False, no_index=False, separator=',', case_sensitive=False, withsuffixtrie=False, index_empty=False)), 'embedding': FlatVectorField(name='embedding', type=<FieldTypes.VECTOR: 'vector'>, path='$.embedding', attrs=FlatVectorFieldAttributes(dims=3, algorithm=<VectorIndexAlgorithm.FLAT: 'FLAT'>, datatype=<VectorDataType.FLOAT32: 'FLOAT32'>, distance_metric=<VectorDistanceMetric.COSINE: 'COSINE'>, initial_cap=None, index_missing=False, block_size=None)), 'user1': TagField(name='u

#### remove_field(field_name)

Removes a field from the schema based on the specified name.

This method is useful for dynamically altering the schema by removing existing fields.

In [5]:
try:
    schema.remove_field("user1")
    print("Tag field 'user1' removed from the schema.")
except Exception as e:
    print(f"Error : {e}")

Tag field 'user1' removed from the schema.


### Index-Level Stopwords Configuration in RedisVL

The `IndexInfo` class in RedisVL allows you to control stopwords at the index level using the `stopwords` field. Stopwords are common words that are filtered out during indexing to improve search efficiency and relevance.

**Configuration Options:**
- `None` (default): Uses Redis's built-in list of ~300 common stopwords.
- `[]` (empty list): Disables stopwords completely (equivalent to `STOPWORDS 0`).
- Custom list: Provide your own list of stopwords, e.g., `["the", "a", "an"]`.

**Example (YAML):**
```yaml
index:
  name: my-index
  prefix: myprefix
  stopwords: ["the", "a", "an"]
```

This configuration ensures that the specified stopwords are filtered out during indexing, improving the quality of your search results.

In [ ]:
schema = IndexSchema.from_dict({
    "index": {
        "name": "my-index",
        "prefix": "myprefix",
        "stopwords": ["the", "a", "an"]
    },
    "fields": [
        {"name": "title", "type": "text"},
        {"name": "embedding", "type": "vector", "attrs": {"dims": 3, "algorithm": "flat", "datatype": "float32"}}
    ]
})

### Basic Field Types in RedisVL

RedisVL supports several field types for defining your index schema. 

Each field type is optimized for different data and search requirements.

- **Text Fields**: Store and index free-form text for full-text search. Example: `{'name': 'title', 'type': 'text'}`
- **Tag Fields**: Store categorical or label data for fast filtering. Example: `{'name': 'category', 'type': 'tag'}`
- **Numeric Fields**: Store numbers for range queries and sorting. Example: `{'name': 'price', 'type': 'numeric'}`
- **Geo Fields**: Store geographic coordinates for location-based queries. Example: `{'name': 'location', 'type': 'geo'}`
- **Vector Fields**: Store high-dimensional vectors for similarity search. Example: `{'name': 'embedding', 'type': 'vector', 'attrs': {...}}`
    - **HNSW Vector Fields**: Use the HNSW (Hierarchical Navigable Small World) algorithm for efficient approximate nearest neighbor search. Specify with `algorithm: 'hnsw'` in the vector field's `attrs`.
    - **FLAT Vector Fields**: Use the FLAT algorithm for exact nearest neighbor search. Specify with `algorithm: 'flat'`.

Each field type can be customized with additional attributes depending on your use case.
- Both can combine vector search with metadata filtering.

### TextField and TextFieldAttributes in RedisVL

A `TextField` in RedisVL is used for full-text search and supports a variety of attributes to control indexing and search behavior.

**Key attributes:**
- `sortable`: Make the field sortable in search results.
- `index_missing`: Index documents even if this field is missing.
- `no_index`: Exclude this field from being indexed (useful for storage only).
- `weight`: Importance of this field in search ranking (default: 1).
- `no_stem`: Disable stemming (words are not reduced to their root form).
- `withsuffixtrie`: Enable suffix trie for fast suffix queries.
- `phonetic_matcher`: Enable phonetic matching (e.g., 'dm:en' for English).
- `index_empty`: Allow indexing/searching for empty strings.
- `unf`: Un-normalized form for sortable fields.

These options allow you to fine-tune how text data is indexed and searched in Redis.

In [8]:
#from redisvl.schema import IndexSchema, TextField, TextFieldAttributes

from redisvl.schema import IndexSchema, TextField
from redisvl.schema.fields import TextFieldAttributes

In [9]:
# Define a TextField with all possible attributes
text_field = TextField(
    name = "description",
    attrs= TextFieldAttributes(
        sortable        = True,         # Allow sorting by this field
        index_missing   = True,         # Index docs even if field is missing
        no_index        = False,        # Field is indexed
        weight          = 2.0,          # Higher importance in ranking
        no_stem         = True,         # Disable stemming
        withsuffixtrie  = True,         # Enable suffix trie for fast suffix search
        phonetic_matcher= "dm:en",      # Enable English phonetic matching
        index_empty     = True,         # Index empty strings
        unf             = True          # Un-normalized form for sorting
    )
)

In [10]:
# Convert the TextField object to a dictionary using .model_dump() (or .dict() for older Pydantic)
text_field_dict = text_field.model_dump()

# Now use this dict in the fields list for IndexSchema.from_dict
schema = IndexSchema.from_dict({
    "index": {
        "name": "my-index",
        "prefix": "myprefix",
        "stopwords": ["the", "a", "an"]
    },
    "fields": [
        text_field_dict,
        {"name": "embedding", "type": "vector", "attrs": {"dims": 3, "algorithm": "flat", "datatype": "float32"}}
    ]
})

print(schema)

index=IndexInfo(name='my-index', prefix='myprefix', key_separator=':', storage_type=<StorageType.HASH: 'hash'>, stopwords=['the', 'a', 'an']) fields={'description': TextField(name='description', type=<FieldTypes.TEXT: 'text'>, path=None, attrs=TextFieldAttributes(sortable=True, index_missing=True, no_index=False, weight=2.0, no_stem=True, withsuffixtrie=True, phonetic_matcher='dm:en', index_empty=True, unf=True)), 'embedding': FlatVectorField(name='embedding', type=<FieldTypes.VECTOR: 'vector'>, path=None, attrs=FlatVectorFieldAttributes(dims=3, algorithm=<VectorIndexAlgorithm.FLAT: 'FLAT'>, datatype=<VectorDataType.FLOAT32: 'FLOAT32'>, distance_metric=<VectorDistanceMetric.COSINE: 'COSINE'>, initial_cap=None, index_missing=False, block_size=None))} version='0.1.0'


#### Tag fields

In [11]:
# Example: Using custom TagField objects with all possible attributes in IndexSchema.from_dict
from redisvl.schema.fields import TagField, TagFieldAttributes

In [13]:
# Define three different TagField objects with various attributes
tag_field1 = TagField(
    name = "category",
    attrs = TagFieldAttributes(
        separator = ",",           # Use comma as separator for multi-value tags
        sortable = True,           # Allow sorting by this field
        index_missing = True,      # Index docs even if field is missing
        no_index = False           # Field is indexed
    )
)

tag_field2 = TagField(
    name = "author",
    attrs = TagFieldAttributes(
        separator = ";",           # Use semicolon as separator
        sortable = False,          # Not sortable
        index_missing = False,     # Do not index if missing
        no_index = False           # Field is indexed
    )
)

tag_field3 = TagField(
    name = "region",
    attrs = TagFieldAttributes(
        separator = "|",            # Use pipe as separator
        sortable = True,            # Allow sorting
        index_missing = True,      # Index docs even if field is missing
        no_index = True            # Field is not indexed (storage only)
    )
)

# Convert TagField objects to dicts
tag_field_dict1 = tag_field1.model_dump()
tag_field_dict2 = tag_field2.model_dump()
tag_field_dict3 = tag_field3.model_dump()

# Use these tag fields in the schema definition
schema = IndexSchema.from_dict({
    "index": {
        "name": "my-index-tags",
        "prefix": "myprefix",
        "stopwords": ["the", "a", "an"]
    },
    "fields": [
        tag_field_dict1,
        tag_field_dict2,
        tag_field_dict3,
        {"name": "embedding", "type": "vector", "attrs": {"dims": 3, "algorithm": "flat", "datatype": "float32"}}
    ]
})

print(schema)

index=IndexInfo(name='my-index-tags', prefix='myprefix', key_separator=':', storage_type=<StorageType.HASH: 'hash'>, stopwords=['the', 'a', 'an']) fields={'category': TagField(name='category', type=<FieldTypes.TAG: 'tag'>, path=None, attrs=TagFieldAttributes(sortable=True, index_missing=True, no_index=False, separator=',', case_sensitive=False, withsuffixtrie=False, index_empty=False)), 'author': TagField(name='author', type=<FieldTypes.TAG: 'tag'>, path=None, attrs=TagFieldAttributes(sortable=False, index_missing=False, no_index=False, separator=';', case_sensitive=False, withsuffixtrie=False, index_empty=False)), 'region': TagField(name='region', type=<FieldTypes.TAG: 'tag'>, path=None, attrs=TagFieldAttributes(sortable=True, index_missing=True, no_index=True, separator='|', case_sensitive=False, withsuffixtrie=False, index_empty=False)), 'embedding': FlatVectorField(name='embedding', type=<FieldTypes.VECTOR: 'vector'>, path=None, attrs=FlatVectorFieldAttributes(dims=3, algorithm=<Ve

### NumericField and NumericFieldAttributes in RedisVL

A `NumericField` in RedisVL is used to store and index numeric values (integers or floats) for range queries, sorting, and filtering. You can control its behavior using `NumericFieldAttributes`.

**Key attributes:**
- `sortable`: (bool) Make the field sortable in search results. Enables sorting by this field in queries.
- `index_missing`: (bool) Index documents even if this field is missing. Useful for sparse data.
- `no_index`: (bool) Exclude this field from being indexed (useful for storage only, not search).

**Attribute explanations:**
- `sortable=True`: Allows you to sort search results by this numeric field (e.g., price, score).
- `index_missing=True`: Ensures documents are indexed even if this field is missing, so you can search for documents where the field is absent.
- `no_index=True`: The field is stored but not indexed, so it cannot be used in search or filter queries (useful for metadata only).

You can combine these attributes to fine-tune how numeric data is indexed and queried in RedisVL.

In [14]:
# Example: Using custom NumericField objects with all possible attributes in IndexSchema.from_dict
from redisvl.schema.fields import NumericField, NumericFieldAttributes

# Define three different NumericField objects with various attributes
numeric_field1 = NumericField(
    name = "price",
    attrs = NumericFieldAttributes(
        sortable = True,            # Allow sorting by this field
        index_missing = True,      # Index docs even if field is missing
        no_index = False           # Field is indexed
    )
)

numeric_field2 = NumericField(
    name = "rating",
    attrs = NumericFieldAttributes(
        sortable = False,           # Not sortable
        index_missing = False,     # Do not index if missing
        no_index = False           # Field is indexed
    )
)

numeric_field3 = NumericField(
    name = "discount",
    attrs = NumericFieldAttributes(
        sortable = True,            # Allow sorting
        index_missing = True,      # Index docs even if field is missing
        no_index = True            # Field is not indexed (storage only)
    )
)

# Convert NumericField objects to dicts
numeric_field_dict1 = numeric_field1.model_dump()
numeric_field_dict2 = numeric_field2.model_dump()
numeric_field_dict3 = numeric_field3.model_dump()

# Use these numeric fields in the schema definition
schema = IndexSchema.from_dict({
    "index": {
        "name": "my-index-numeric",
        "prefix": "myprefix",
        "stopwords": ["the", "a", "an"]
    },
    "fields": [
        numeric_field_dict1,
        numeric_field_dict2,
        numeric_field_dict3,
        {"name": "embedding", "type": "vector", "attrs": {"dims": 3, "algorithm": "flat", "datatype": "float32"}}
    ]
})

print(schema)

index=IndexInfo(name='my-index-numeric', prefix='myprefix', key_separator=':', storage_type=<StorageType.HASH: 'hash'>, stopwords=['the', 'a', 'an']) fields={'price': NumericField(name='price', type=<FieldTypes.NUMERIC: 'numeric'>, path=None, attrs=NumericFieldAttributes(sortable=True, index_missing=True, no_index=False, unf=False)), 'rating': NumericField(name='rating', type=<FieldTypes.NUMERIC: 'numeric'>, path=None, attrs=NumericFieldAttributes(sortable=False, index_missing=False, no_index=False, unf=False)), 'discount': NumericField(name='discount', type=<FieldTypes.NUMERIC: 'numeric'>, path=None, attrs=NumericFieldAttributes(sortable=True, index_missing=True, no_index=True, unf=False)), 'embedding': FlatVectorField(name='embedding', type=<FieldTypes.VECTOR: 'vector'>, path=None, attrs=FlatVectorFieldAttributes(dims=3, algorithm=<VectorIndexAlgorithm.FLAT: 'FLAT'>, datatype=<VectorDataType.FLOAT32: 'FLOAT32'>, distance_metric=<VectorDistanceMetric.COSINE: 'COSINE'>, initial_cap=Non

### Why Does Redis Provide Database-like Features for Vector Search?

Many vector databases (like ChromaDB, FAISS, Pinecone) allow you to store text, embeddings, and metadata, but Redis takes a broader approach by providing database-like features. Here’s the rationale:

**1. Redis is a General-purpose, Multi-model Database:**
- Redis is not just a vector store; it’s a high-performance, in-memory database supporting multiple data types (strings, hashes, sets, lists, streams, JSON, and vectors).
- This allows you to combine vector search with other data models and operations in a single system.

**2. Namespaces and Logical Databases:**
- Redis supports logical databases (numbered 0, 1, 2, ...) to isolate key spaces. This is useful for multi-tenant applications, development vs. production separation, or organizing data by use case.
- Key prefixes and logical DBs help simulate namespaces, making it easier to manage and organize data at scale.

**3. Metadata and Hybrid Search:**
- Redis natively supports storing metadata alongside vectors (using hashes or JSON).
- You can perform hybrid queries: filter by metadata (e.g., tags, numbers, geo) and then do vector similarity search—all in one query.

**4. Real-time Performance and Flexibility:**
- Redis is designed for ultra-low latency and high throughput, making it ideal for real-time AI/ML applications.
- You can update, delete, and search data instantly, and combine vector search with other Redis features (pub/sub, streams, caching, etc.).

**5. Extensibility via Modules:**
- Redis modules (like Redisearch) add advanced features such as full-text search, secondary indexes, and vector search, turning Redis into a powerful multi-modal database.

**Summary:**
Redis provides database-like features for vector search to offer flexibility, scalability, and real-time performance, while allowing you to combine vector, metadata, and other data types in a single, unified system. This makes Redis suitable for a wide range of production AI and search workloads beyond what specialized vector stores alone can offer.

#### Vector Field Types and Attributes in RedisVL

Vector fields enable semantic similarity search using various algorithms. 

All vector fields share a set of common attributes, with some algorithm-specific options. 

Below is a summary of the key attributes and their usage, with examples.

**Common Vector Field Attributes:**
- `dims` *(int)*: Dimensionality of the vector embeddings. Example: `dims=1536` for OpenAI embeddings.
- `algorithm` *(VectorIndexAlgorithm)*: Indexing algorithm for vector search. Options:
    - `'flat'`: Brute-force exact search. 100% recall, slower for large datasets. Best for <10K vectors.
    - `'hnsw'`: Graph-based approximate search. Fast with high recall (95-99%). Best for general use.
    - `'svs-vamana'`: Scalable Vector Search with VAMANA graph algorithm. Fast, memory-efficient, supports compression (float16/float32 only). Optimized for Intel hardware.
- `datatype` *(VectorDataType)*: Float precision for the vector embeddings. Options:
    - `'float32'` (default, most common)
    - `'float16'`, `'bfloat16'`, `'float64'` (SVS-VAMANA: only float16/float32)
- `distance_metric` *(VectorDistanceMetric)*: Similarity metric for search. Options:
    - `'COSINE'` (default, most common for semantic search)
    - `'L2'` (Euclidean distance)
    - `'IP'` (Inner Product)
- `initial_cap` *(int, optional)*: Initial capacity hint for memory allocation. Example: `initial_cap=10000` for 10,000 vectors.
- `index_missing` *(bool, optional)*: If `True`, allows indexing/searching for documents missing this field. Useful for sparse data.



**Example: Basic Vector Field (HNSW, float32, cosine)**
```python
from redisvl.schema.fields import VectorField, VectorFieldAttributes

vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=768,
        algorithm="hnsw",
        datatype="float32",
        distance_metric="cosine",
        initial_cap=10000,
        index_missing=False
    )
)
```


**Example: Flat Vector Field (exact search, float32, L2)**
```python
vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=1536,
        algorithm="flat",
        datatype="float32",
        distance_metric="l2"
    )
)
```



**Example: SVS-VAMANA Vector Field (approximate, float16, IP)**
```python
vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=1024,
        algorithm="svs-vamana",
        datatype="float16",
        distance_metric="ip",
        initial_cap=50000
    )
)
```




**Notes:**
- Choose `flat` for small datasets or when exact recall is required.
- Use `hnsw` for most production workloads (fast, high recall, scalable).
- Use `svs-vamana` for large-scale, memory-efficient search (Intel hardware, float16/float32 only).
- Set `initial_cap` to optimize memory allocation for expected dataset size.
- `index_missing=True` is useful for hybrid or sparse data scenarios.

For more details, see the RedisVL documentation on [Vector Field Algorithms and Attributes](https://redisvl.readthedocs.io/en/latest/).

### FLAT Index in Vector Search (RedisVL)

The **FLAT** index is a brute-force, exact nearest neighbor search algorithm for vector fields. It is simple, highly accurate, and best suited for small to medium datasets.

#### How FLAT Index Works:
- Every query compares the input vector to **all** stored vectors in the index.
- Computes the distance (cosine, L2, or inner product) between the query and each vector.
- Returns the top-k most similar vectors (exact results, 100% recall).

#### When to Use FLAT Index:
- Best for datasets with **<10,000 vectors** (small to moderate size).
- When you need **exact** results (no approximation).
- Useful for testing, validation, or when accuracy is more important than speed.
- Not recommended for very large datasets (scales linearly, can be slow).

#### FLAT Index Illustration:

```
Query Vector
    |
    v
+-------------------+
|   Vector Index    |
|-------------------|
| [v1]              |
| [v2]              |
| [v3]              |
| ...               |
| [vn]              |
+-------------------+
    |
    v
Compare query to every vector (brute-force)
    |
    v
Return top-k most similar vectors (exact)
```

#### Example: Defining a FLAT Vector Field in RedisVL
```python
from redisvl.schema.fields import VectorField, VectorFieldAttributes

vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=1536,
        algorithm="flat",
        datatype="float32",
        distance_metric="cosine"
    )
)
```

#### Pros and Cons of FLAT Index:
- **Pros:**
    - 100% recall (exact results)
    - Simple to implement and understand
    - No approximation or randomness
- **Cons:**
    - Slower for large datasets (linear scan)
    - High memory and compute cost as data grows

**Summary:** Use FLAT index for small datasets or when you need exact, brute-force vector search. For larger datasets, consider approximate algorithms like HNSW or SVS-VAMANA.

### HNSW Index in Vector Search (RedisVL)

The **HNSW** (Hierarchical Navigable Small World) index is a graph-based, approximate nearest neighbor search algorithm. It is the default and most popular choice for large-scale, high-performance vector search in RedisVL and many other vector databases.

#### How HNSW Index Works:
- Organizes vectors in a multi-layered, navigable small-world graph.
- Search starts at the top layer and quickly navigates down to the closest vectors using graph links.
- Returns the top-k most similar vectors with high recall (typically 95-99%).
- Provides a balance between speed, memory usage, and accuracy.

#### When to Use HNSW Index:
- Best for **large datasets** (10,000+ vectors, scales to millions).
- When you need **fast** and **scalable** approximate search with high recall.
- Suitable for most production workloads and real-time applications.

#### HNSW Index Illustration:

```
Query Vector
    |
    v
+-------------------+
|   HNSW Graph      |
|-------------------|
| [v1]---[v2]---[v3]|
|   |      |      | |
|  ...   ...   ...  |
+-------------------+
    |
    v
Navigate graph layers to find nearest neighbors (approximate)
    |
    v
Return top-k most similar vectors (high recall)
```

#### Example: Defining an HNSW Vector Field in RedisVL
```python
from redisvl.schema.fields import VectorField, VectorFieldAttributes

vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=1536,
        algorithm="hnsw",
        datatype="float32",
        distance_metric="cosine",
        initial_cap=100000
    )
)
```

#### Pros and Cons of HNSW Index:
- **Pros:**
    - Extremely fast for large datasets
    - High recall (95-99%)
    - Scalable and memory-efficient
    - Widely adopted and well-tested
- **Cons:**
    - Results are approximate (not 100% exact)
    - Slightly more complex to tune than FLAT

**Summary:** Use HNSW index for most real-world, large-scale vector search applications where speed and scalability are critical, and near-exact recall is acceptable.

#### How HNSW Builds Small World Graphs

The "small world" in HNSW refers to a graph structure where most nodes (vectors) can be reached from every other by a small number of steps, thanks to both short-range and long-range connections.

**How are small worlds formed in HNSW?**
- **Multi-layered Graph:** HNSW builds several layers of graphs. The top layers have fewer nodes and more long-range links; lower layers have more nodes and short-range links.
- **Randomized Layer Assignment:** Each new vector is randomly assigned to a certain number of layers (higher layers are sparser).
- **Linking:** When a new vector is added, it is connected to its nearest neighbors in each layer it belongs to. This creates both local (short-range) and global (long-range) connections.
- **Navigation:** During search, the algorithm starts at the top layer (with long-range links) and quickly "zooms in" to the closest region by following links down to lower layers (with more local links).

**Illustration:**

```
Layer 3 (few nodes, long links):
   [A]------[B]
     \      /
      [C]

Layer 2:
   [A]---[B]---[C]
    |     |     |
   [D]---[E]---[F]

Layer 1 (many nodes, short links):
   [A]-[B]-[C]-[D]-[E]-[F]-[G]-[H]
```
- Higher layers: sparse, long-range connections (fast global navigation)
- Lower layers: dense, short-range connections (precise local search)

**Summary:**
- HNSW's "small world" property means you can reach any vector quickly by combining a few long jumps (top layers) and many short hops (bottom layers).
- This structure enables fast, scalable, and accurate approximate nearest neighbor search.

#### Is HNSW Index Construction Time-Consuming?

- **HNSW index construction is generally slower than FLAT,** because it must build and maintain a multi-layered graph with both short- and long-range links for every new vector.
- Each insertion involves finding nearest neighbors and updating connections in multiple layers, which is more complex than simply appending to a list (as in FLAT).

**However:**
- HNSW is designed for **incremental, online building**—you can add vectors at any time, and the index updates efficiently without needing a full rebuild.
- For most practical dataset sizes, HNSW build time is reasonable and is a one-time or background cost. The search speed and scalability benefits far outweigh the extra build time for large datasets.
- In production, HNSW is the preferred choice for fast, scalable search, even if it takes longer to build than FLAT.

**Summary:**
- HNSW takes longer to build than FLAT, but this is usually acceptable for large-scale, high-performance vector search.
- The trade-off: **slower build, much faster search** (especially as data grows).

### SVS-VAMANA Index in Vector Search (RedisVL)

The **SVS-VAMANA** index is a scalable, graph-based approximate nearest neighbor (ANN) algorithm designed for high performance and memory efficiency, especially on large datasets and modern hardware (notably Intel CPUs).


#### What is SVS-VAMANA?
- **SVS** stands for Scalable Vector Search.
- **VAMANA** is the underlying graph algorithm, similar in spirit to HNSW but optimized for speed, memory, and parallelism.
- Supports compressed vector types (float16, float32) for reduced memory usage.
- Available in Redis with the Redisearch module (and RedisVL) for advanced, large-scale vector search.

#### How SVS-VAMANA Works:
- Builds a navigable proximity graph (like HNSW) but with optimizations for memory and search speed.
- Uses a single-layer graph (not multi-layered like HNSW) with carefully chosen connections for fast traversal.
- Supports SIMD/vectorized operations and parallel search for high throughput.
- Designed for incremental, online updates (add/remove vectors efficiently).

#### SVS-VAMANA Index Illustration:
```
Query Vector
    |
    v
+-------------------+
|  VAMANA Graph     |
|-------------------|
| [v1]--[v2]--[v3]  |
|  |   /   \   |    |
| [v4]--[v5]--[v6]  |
+-------------------+
    |
    v
Traverse graph edges to find nearest neighbors (approximate)
    |
    v
Return top-k most similar vectors (high recall)
```

#### When to Use SVS-VAMANA:
- Best for **very large datasets** (millions of vectors).
- When you need **fast, memory-efficient** approximate search.
- When running on Intel CPUs (optimized for AVX512, etc.).
- If you want to use compressed vector types (float16/float32).

#### Example: Defining an SVS-VAMANA Vector Field in RedisVL
```python
from redisvl.schema.fields import VectorField, VectorFieldAttributes

vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=1024,
        algorithm="svs-vamana",
        datatype="float16",  # or "float32"
        distance_metric="cosine",
        initial_cap=500000
    )
)
```

#### Pros and Cons of SVS-VAMANA:
- **Pros:**
    - Extremely fast and scalable for large datasets
    - Lower memory usage (supports float16)
    - Parallel and SIMD-optimized (great for modern CPUs)
    - Incremental updates supported
- **Cons:**
    - Only available in recent Redis/Redisearch versions
    - Best performance on Intel hardware
    - Results are approximate (like HNSW)

**Summary:** Use SVS-VAMANA for massive, production-scale vector search where speed, memory efficiency, and hardware optimization are critical. For most general use cases, HNSW is sufficient, but SVS-VAMANA is ideal for the largest and most demanding workloads.


### Comparison of Indexing Algorithms: RedisVL, FAISS, ChromaDB, Weaviate, Pinecone

Below is a comparison of the major vector databases and libraries in terms of their supported vector indexing algorithms:

| System      | FLAT/Brute-Force | HNSW | IVF/IVFFlat | PQ/IVFPQ | Annoy | SVS-VAMANA | ScaNN | Proprietary/Other |
|-------------|:----------------:|:----:|:-----------:|:--------:|:-----:|:----------:|:-----:|:----------------:|
| **RedisVL** |   ✅ (FLAT)       | ✅   |      ❌      |    ❌     |  ❌   |    ✅       |  ❌   |   ❌             |
| **FAISS**   |   ✅ (Flat)       | ✅   |   ✅ (IVF)   |   ✅      |  ❌   |    ❌       |  ✅   |   ❌             |
| **ChromaDB**|   ✅ (Flat)       | ✅   |      ❌      |    ❌     |  ❌   |    ❌       |  ❌   |   ❌             |
| **Weaviate**|   ✅ (Flat)       | ✅   |      ❌      |    ❌     |  ❌   |    ❌       |  ❌   |   ❌             |
| **Pinecone**|   ✅ (Flat)       | ✅   |      ❌      |    ❌     |  ❌   |    ❌       |  ❌   |   ✅ (Proprietary)|

#### Key:
- **FLAT/Brute-Force:** Exact search, linear scan (sometimes called Flat or Dense).
- **HNSW:** Hierarchical Navigable Small World, fast approximate search.
- **IVF/IVFFlat:** Inverted File Index, partitions vectors for faster search (FAISS only).
- **PQ/IVFPQ:** Product Quantization, compresses vectors for memory efficiency (FAISS only).
- **Annoy:** Approximate Nearest Neighbors Oh Yeah (Spotify, not used in these DBs).
- **SVS-VAMANA:** Scalable Vector Search, VAMANA graph (RedisVL only).
- **ScaNN:** Google’s fast ANN library (FAISS only, via integration).
- **Proprietary/Other:** Custom or closed-source algorithms (Pinecone uses its own).

#### Notes:
- **RedisVL:** Supports FLAT (exact), HNSW (approximate), and SVS-VAMANA (approximate, Intel-optimized).
- **FAISS:** Most flexible; supports Flat, HNSW, IVF, PQ, IVFPQ, and ScaNN (with advanced configuration).
- **ChromaDB:** Supports Flat and HNSW (via underlying libraries, e.g., FAISS or Annoy backend).
- **Weaviate:** Supports Flat and HNSW (HNSW is default for vector search).
- **Pinecone:** Supports Flat and HNSW, but also uses proprietary, cloud-optimized ANN algorithms for performance and scaling.

**Summary:**
- If you need the widest range of algorithms and fine-tuning, **FAISS** is the most advanced.
- For production, cloud, and managed services, **Pinecone** and **Weaviate** offer simplicity and scalability, but with fewer algorithm choices.
- **RedisVL** is unique in supporting SVS-VAMANA and is strong for real-time, in-memory workloads.
- **ChromaDB** is simple and easy to use, with Flat and HNSW support for most use cases.


### HNSW Vector Fields in RedisVL

HNSW (Hierarchical Navigable Small World) is a graph-based, approximate nearest neighbor (ANN) algorithm. In RedisVL, you can define a vector field to use HNSW for fast, scalable vector search.

**Key Points:**
- HNSW is the default and most popular ANN algorithm in RedisVL.
- It is highly efficient for large datasets (10,000+ vectors, scales to millions).
- Provides high recall (95-99%) and very fast search times.
- Supports incremental, online updates (add vectors at any time).

**Defining an HNSW Vector Field:**
```python
from redisvl.schema.fields import VectorField, VectorFieldAttributes

vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=1536,  # Set to your embedding dimension
        algorithm="hnsw",
        datatype="float32",  # or "float16" if supported
        distance_metric="cosine",  # or "l2", "ip"
        initial_cap=100000  # Optional: expected max vectors
    )
)
```

**Tuning HNSW Parameters:**
- `dims`: The dimension of your vectors (must match your embeddings).
- `algorithm`: Set to `'hnsw'` for HNSW indexing.
- `datatype`: Use `'float32'` for most models; `'float16'` for memory savings if supported.
- `distance_metric`: `'cosine'` (semantic), `'l2'` (Euclidean), or `'ip'` (inner product).
- `initial_cap`: (Optional) Pre-allocate memory for large datasets.

**Advanced HNSW Parameters (if supported):**
- `M`: Number of bi-directional links per node (higher = more accuracy, more memory).
- `ef_construction`: Controls index build accuracy/speed (higher = better recall, slower build).
- `ef_search`: Controls search accuracy/speed (higher = better recall, slower search).

**Example with advanced parameters:**
```python
vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=768,
        algorithm="hnsw",
        datatype="float32",
        distance_metric="cosine",
        initial_cap=50000,
        M=16,  # default: 16
        ef_construction=200,  # default: 200
        ef_search=40  # default: 40
    )
)
```

**Summary:**
- Use HNSW vector fields for fast, scalable, and high-recall vector search in RedisVL.
- Tune `M`, `ef_construction`, and `ef_search` for your accuracy/speed trade-off.
- HNSW is suitable for most production workloads and is the recommended default for large-scale vector search.


#### HNSW Vector Field Example: Balanced Configuration

A recommended starting point for HNSW in RedisVL is a balanced configuration, which provides a good trade-off between search speed, recall, and memory usage.

**YAML Example:**
```yaml
- name: embedding
  type: vector
  attrs:
    algorithm: hnsw
    dims: 768
    distance_metric: cosine
    datatype: float32
    # Index-time parameters (set during index creation)
    m: 16                    # Graph connectivity (default: 16)
    ef_construction: 200     # Build-time accuracy (default: 200)
    # Note: ef_runtime can be set at query time via VectorQuery
```

**Python Example:**
```python
from redisvl.schema.fields import VectorField, VectorFieldAttributes

vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=768,
        algorithm="hnsw",
        datatype="float32",
        distance_metric="cosine",
        M=16,  # Graph connectivity
        ef_construction=200  # Build-time accuracy
        # ef_search (runtime) can be set at query time
    )
)
```

**Parameter Notes:**
- `M` (or `m` in YAML): Controls the number of bi-directional links per node. Higher values increase accuracy and memory usage.
- `ef_construction`: Controls the accuracy and speed of index building. Higher values improve recall but slow down index creation.
- `ef_search` (or `ef_runtime`): Controls search accuracy at query time. Set this in your query, not in the schema.

**Tip:**
- Start with these defaults and tune `M`, `ef_construction`, and `ef_search` based on your dataset size and recall/speed needs.


#### HNSW Vector Field Example: Balanced Configuration

A recommended starting point for HNSW in RedisVL is a balanced configuration, which provides a good trade-off between search speed, recall, and memory usage.

**YAML Example:**
```yaml
- name: embedding
  type: vector
  attrs:
    algorithm: hnsw
    dims: 768
    distance_metric: cosine
    datatype: float32
    # Index-time parameters (set during index creation)
    m: 16                    # Graph connectivity (default: 16)
    ef_construction: 200     # Build-time accuracy (default: 200)
    # Note: ef_runtime can be set at query time via VectorQuery
```

**Python Example:**
```python
from redisvl.schema.fields import VectorField, VectorFieldAttributes

vector_field = VectorField(
    name="embedding",
    attrs=VectorFieldAttributes(
        dims=768,
        algorithm="hnsw",
        datatype="float32",
        distance_metric="cosine",
        M=16,  # Graph connectivity
        ef_construction=200  # Build-time accuracy
        # ef_search (runtime) can be set at query time
    )
)
```

**Parameter Notes:**
- `M` (or `m` in YAML): Controls the number of bi-directional links per node. Higher values increase accuracy and memory usage.
- `ef_construction`: Controls the accuracy and speed of index building. Higher values improve recall but slow down index creation.
- `ef_search` (or `ef_runtime`): Controls search accuracy at query time. Set this in your query, not in the schema.

**Tip:**
- Start with these defaults and tune `M`, `ef_construction`, and `ef_search` based on your dataset size and recall/speed needs.


---
### SearchIndex in RedisVL
---

A `SearchIndex` in RedisVL is the main abstraction for managing and querying a vector (or hybrid) search index in Redis. 

It encapsulates the schema, index creation, data insertion, and search operations, making it easy to work with Redis as a vector database from Python.

**Key Features:**
- Manages the lifecycle of a Redisearch index (create, drop, update).
- Handles both vector and metadata fields as defined in your schema.
- Provides methods for inserting, updating, and deleting documents.
- Supports vector similarity search, hybrid search (vector + metadata), and filtering.

**Typical Workflow:**
1. **Define a schema** (using `IndexSchema` and field classes).
2. **Create a SearchIndex** object with the schema and Redis connection.
3. **Create the index** in Redis (if not already present).
4. **Insert documents** (with vectors and metadata).
5. **Query** using vector search, hybrid search, or filters.

In [1]:
import redis
print(redis.__version__)

7.1.0


In [3]:
!pip install --upgrade redis

  Attempting uninstall: redis
    Found existing installation: redis 3.5.3
    Uninstalling redis-3.5.3:
      Successfully uninstalled redis-3.5.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langflow 1.6.9 requires redis<6.0.0,>=5.2.1, but you have redis 7.1.0 which is incompatible.
redisvl 0.12.1 requires redis<7.0,>=5.0, but you have redis 7.1.0 which is incompatible.
redis-py-cluster 2.1.3 requires redis<4.0.0,>=3.0.0, but you have redis 7.1.0 which is incompatible.


In [2]:
from redisvl.index import SearchIndex

In [3]:
# Initialize the index object with schema from file
index = SearchIndex.from_yaml(
    "schemas/schema_test.yaml",                              # Path to your YAML schema file
    redis_url       = "redis://localhost:6379", # Redis connection URL
    validate_on_load= True                      # Validate schema on load
)

In [4]:
# Create the index in Redis
index.create(overwrite=True, drop=False)

In [5]:
# Example: Defining 'data' for index.load(data)
# Each dictionary should match your schema fields (e.g., id, title, embedding)

import numpy as np

# Example with two documents and random embeddings (replace with your real data)
data = [
    {
        "id": "doc1",
        "title": "Redis Vector Search",
        "embedding": np.random.rand(768).tolist()  # 768-dim vector
    },
    {
        "id": "doc2",
        "title": "Another Document",
        "embedding": np.random.rand(768).tolist()
    }
    # Add more documents as needed
]

In [6]:
# Load data into the index
# 'data' should be an iterable of dictionaries, each representing a document
index.load(data)

['doc:01KBVWXF3EPXQHJ2P8173X8PQV', 'doc:01KBVWXF4BYTGJX5WD201J9QRQ']

In [7]:
# Delete the index and all associated data
index.delete(drop=True)

---
## Example: Creating and Using a SearchIndex in RedisVL
---

In [ ]:
# We'll need to install the Redis client
#!pip install redis

#Install wget to pull zip file
#!pip install wget

In [ ]:
#pip install wget

In [9]:
!pip install wget

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9712 sha256=57f571a188ef964ace9a7458edcbf2187f8a444b41dd2ac598a3507553d3dbd3
  Stored in directory: c:\users\reach\appdata\local\pip\cache\wheels\8a\b8\04\0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget


In [10]:
import openai

from typing import List, Iterator
import pandas as pd
import numpy as np
import os
import wget
from ast import literal_eval

# Redis client library for Python
import redis

# I've set this to our new embeddings model, this can be changed to the embedding model of your choice
EMBEDDING_MODEL = "text-embedding-3-small"

# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [11]:
article_df = pd.read_csv(r"D:\Makesh\Working\AI\RPS\Day09\vector_database_wikipedia_articles_embedded.csv")

In [12]:
article_df.head()

,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4


In [13]:
# Read vectors from strings back into a list
article_df['title_vector']   = article_df.title_vector.apply(literal_eval)
article_df['content_vector'] = article_df.content_vector.apply(literal_eval)

# Set vector_id to be a string
article_df['vector_id'] = article_df['vector_id'].apply(str)

In [ ]:
article_df.info(show_counts=True)

In [14]:
import redis

from redis.commands.search.query import Query
from redis.commands.search.field import (
    TextField,
    VectorField
)

from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.field import TextField, VectorField
from redis.commands.search.query import Query


In [15]:
REDIS_HOST =  "localhost"
REDIS_PORT = 6379
REDIS_PASSWORD = "" # default for passwordless Redis


In [16]:
# Connect to Redis
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD
)

In [17]:
redis_client.ping()

True

#### Creating a Search Index

The below cells will show how to specify and create a search index in Redis. We will:

- Set some constants for defining our index like the distance metric and the index name
- Define the index schema with RediSearch fields
- Create the index

In [18]:
# Constants
VECTOR_DIM = len(article_df['title_vector'][0]) # length of the vectors
VECTOR_NUMBER = len(article_df)                 # initial number of vectors
INDEX_NAME = "embeddings-index"                 # name of the search index
PREFIX = "doc"                                  # prefix for the document keys
DISTANCE_METRIC = "COSINE"                      # distance metric for the vectors (ex. COSINE, IP, L2)

In [19]:
# Define RediSearch fields for each of the columns in the dataset
title = TextField(name="title")
url   = TextField(name="url")
text  = TextField(name="text")

title_embedding = VectorField("title_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
text_embedding = VectorField("content_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)

fields = [title, url, text, title_embedding, text_embedding]

In [20]:
# Check if index exists
try:
    redis_client.ft(INDEX_NAME).info()
    print("Index already exists")
except:
    # Create RediSearch Index
    redis_client.ft(INDEX_NAME).create_index(
        fields = fields,
        definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
    )

#### Load Documents into the Index

Now that we have a search index, we can load documents into it. We will use the same documents we used in the previous examples. 

In Redis, either the Hash or JSON (if using RedisJSON in addition to RediSearch) data types can be used to store documents. 

We will use the HASH data type in this example. The below cells will show how to load documents into the index.

In [21]:
def index_documents(client: redis.Redis, prefix: str, documents: pd.DataFrame):
    records = documents.to_dict("records")
    for doc in records:
        key = f"{prefix}:{str(doc['id'])}"

        # create byte vectors for title and content
        title_embedding = np.array(doc["title_vector"], dtype=np.float32).tobytes()
        content_embedding = np.array(doc["content_vector"], dtype=np.float32).tobytes()

        # replace list of floats with byte vectors
        doc["title_vector"] = title_embedding
        doc["content_vector"] = content_embedding

        client.hset(key, mapping = doc)

In [22]:
index_documents(redis_client, PREFIX, article_df)
print(f"Loaded {redis_client.info()['db0']['keys']} documents in Redis search index with name: {INDEX_NAME}")

Loaded 50000 documents in Redis search index with name: embeddings-index


#### Running Search Queries

Now that we have a search index and documents loaded into it, we can run search queries. Below we will provide a function that will run a search query and return the results. Using this function we run a few queries that will show how you can utilize Redis as a vector database. Each example will demonstrate specific features to keep in mind when developing your search application with Redis.

**Return Fields:** You can specify which fields you want to return in the search results. This is useful if you only want to return a subset of the fields in your documents and doesn't require a separate call to retrieve documents. In the below example, we will only return the title field in the search results.

**Hybrid Search:** You can combine vector search with any of the other RediSearch fields for hybrid search such as full text search, tag, geo, and numeric. In the below example, we will combine vector search with full text search.

In [23]:
from openai import OpenAI

In [24]:
def search_redis(
    redis_client: redis.Redis,
    user_query: str,
    index_name: str = "embeddings-index",
    vector_field: str = "title_vector",
    return_fields: list = ["title", "url", "text", "vector_score"],
    hybrid_fields = "*",
    k: int = 20,
) -> List[dict]:

    client = OpenAI()
    
    # Creates embedding vector from user query
    embedded_query = client.embeddings.create(input=user_query,
                                            model=EMBEDDING_MODEL,
                                            ).data[0].embedding

    # Prepare the Query
    base_query = f'{hybrid_fields}=>[KNN {k} @{vector_field} $vector AS vector_score]'
    query = (
        Query(base_query)
         .return_fields(*return_fields)
         .sort_by("vector_score")
         .paging(0, k)
         .dialect(2)
    )
    params_dict = {"vector": np.array(embedded_query).astype(dtype=np.float32).tobytes()}

    # perform vector search
    results = redis_client.ft(index_name).search(query, params_dict)
    for i, article in enumerate(results.docs):
        score = 1 - float(article.vector_score)
        print(f"{i}. {article.title} (Score: {round(score ,3) })")
    return results.docs

In [25]:
# For using OpenAI to generate query embedding

results = search_redis(redis_client, 'modern art in Europe', k=10)

17:27:59 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
0. General Dynamics F-16 Fighting Falcon (Score: 0.034)
1. Mikoyan-Gurevich MiG-17 (Score: 0.033)
2. The Good, the Bad and the Ugly (Score: 0.028)
3. Genestrerio (Score: 0.028)
4. Mikoyan-Gurevich MiG-15 (Score: 0.026)
5. Musical genre (Score: 0.025)
6. Mikoyan-Gurevich MiG-21 (Score: 0.025)
7. Edvard Grieg (Score: 0.024)
8. Licensed to Ill (Score: 0.023)
9. Grumman F4F Wildcat (Score: 0.023)


In [26]:
results = search_redis(redis_client, 'Famous battles in Scottish history', vector_field='content_vector', k=10)

17:28:07 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
0. 585 BC (Score: 0.048)
1. Order of the British Empire (Score: 0.045)
2. 40s BC (Score: 0.042)
3. Order of the Bath (Score: 0.041)
4. Julius Caesar (Score: 0.041)
5. The Convent (Gibraltar) (Score: 0.04)
6. Washington's Birthday (Score: 0.038)
7. 480 (Score: 0.038)
8. 32 (Score: 0.036)
9. 14 BC (Score: 0.036)


Hybrid Queries with Redis

The previous examples showed how run vector search queries with RediSearch. In this section, we will show how to combine vector search with other RediSearch fields for hybrid search. In the below example, we will combine vector search with full text search.

In [27]:
def create_hybrid_field(field_name: str, value: str) -> str:
    return f'@{field_name}:"{value}"'

In [ ]:
# First apply the hybrid filter to only include results with Scottish in the title 
# and then search the title vector for articles about famous battles in Scottish history

results = search_redis(redis_client,
                       "Famous battles in scottish history",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("title", "scottish")
                       )

18:03:12 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
0. Scottish Premier League (Score: 0.01)
1. List of Scottish monarchs (Score: 0.006)
2. Scottish (Score: 0.004)
3. Scottish Borders (Score: -0.001)
4. Scottish Socialist Party (Score: -0.004)


In [ ]:
results

In [34]:
# run a hybrid query for articles about Art in the title vector and only include results with the phrase "Leonardo da Vinci" in the text
results = search_redis(redis_client,
                       "Art",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("text", "Leonardo da Vinci")
                       )


18:03:27 httpx INFO   HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
0. Angel (Score: 0.007)
1. The Da Vinci Code (Score: 0.004)
2. May 28 (Score: -0.0)
3. Po (river) (Score: -0.001)
4. Leonardo da Vinci (Score: -0.001)


In [30]:
# find specific mention of Leonardo da Vinci in the text that our full-text-search query returned
mention = [sentence for sentence in results[0].text.split("\n") if "Leonardo da Vinci" in sentence][0]
mention

'The same cherubim creatures were said to be cast in gold on top of the Ark of the Covenant. Casting metal is one of the oldest forms of artwork, and was attempted by Leonardo da Vinci.'